# Assignment 2: training pipeline

You're using a notebook for this stage. If you'd rather use a script,
write `train.py` instead and delete this file. `main.py` refuses to run if
it finds both `train.py` and `train.ipynb`, so exactly one of them must exist.

Replace this cell and everything below it with your own code. When run
via `uv run python main.py train`, this notebook must, start to finish, with no
manual steps in between:

- Get your Roboflow API key via `from api_key import load_api_key`, so that it
  works whether the key is in your `.env` file or in the environment, and
  download your dataset version from Roboflow Universe.
- Finetune yolo11n/yolo11s (or the -cls variant) on that dataset, from the
  pretrained checkpoint.
- Evaluate on the test split against the two baselines named in the brief: the
  pretrained COCO model zero-shot, and a trivial predict-nothing / majority
  class baseline.
- Sweep the confidence threshold, compute the expected cost at each point from
  the cost matrix you defined in REPORT.md, and save the sweep.
- Save everything `app.py` needs to build its dashboard without retraining:
  the metrics, the per-class numbers, the cost sweep, and the chosen operating
  point.

This is your real training pipeline. It takes as long as it takes, and it runs
wherever training is practical for you: your own machine, a GPU you have access
to, or a hosted notebook. Nothing re-runs it to check your submission.

The weights it produces are what you commit to `weights/` and what your app
serves. Say in REPORT.md where you trained and what settings you used.

Save the trained model into `weights/`. That is the file you commit and the
file your app loads.

How you structure the code beyond that is your call to make and be able to
defend: file layout, function boundaries, naming.


In [2]:
import glob
import json
from collections import Counter, defaultdict
from pathlib import Path

import yaml
from roboflow import Roboflow

from api_key import load_api_key

WORKSPACE = "yolov8-z6snb"
PROJECT = "damage-detect-pzgeu"
VERSION = 1

def download():
    rf = Roboflow(api_key=load_api_key())
    ds = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download(
        "yolov11", location="data/damage-detect-v1", overwrite=True
    )
    root = Path(ds.location)
    print("Dataset folder:", root, "->", sorted(p.name for p in root.iterdir()))
    return root


def dataset_stats(root):
    names = yaml.safe_load(open(root / "data.yaml"))["names"]
    stats = {}
    for split in ["train", "valid", "test"]:
        files = glob.glob(str(root / split / "labels" / "*.txt"))
        boxes, areas = Counter(), defaultdict(list)
        damaged = clean_only = empty = 0
        for f in files:
            classes = set()
            for line in open(f):
                parts = line.split()
                if len(parts) < 5:
                    continue
                c = int(parts[0])
                boxes[names[c]] += 1
                areas[names[c]].append(float(parts[3]) * float(parts[4]))
                classes.add(names[c])
            if "Damage-detected" in classes:
                damaged += 1
            elif classes:
                clean_only += 1
            else:
                empty += 1
        stats[split] = {
            "images": len(files),
            "boxes_per_class": dict(boxes),
            "images_with_damage_box": damaged,
            "images_undamaged_only": clean_only,
            "images_with_no_boxes": empty,
            "median_box_area_pct": {
                k: round(sorted(v)[len(v) // 2] * 100, 1) for k, v in areas.items()
            },
        }
    return stats


root = download()
stats = dataset_stats(root)
Path("results").mkdir(exist_ok=True)
json.dump(stats, open("results/dataset_stats.json", "w"), indent=2)
print(json.dumps(stats, indent=2))

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to data/damage-detect-v1 in yolov11:: 100%|██████████| 4151/4151 [00:01<00:00, 2774.78it/s]


Dataset folder: /workspaces/daiist-assignment-2/data/damage-detect-v1 -> ['README.dataset.txt', 'README.roboflow.txt', 'data.yaml', 'test', 'train', 'valid']
{
  "train": {
    "images": 1501,
    "boxes_per_class": {
      "Damage-detected": 717,
      "Undamaged": 895
    },
    "images_with_damage_box": 698,
    "images_undamaged_only": 803,
    "images_with_no_boxes": 0,
    "median_box_area_pct": {
      "Damage-detected": 62.1,
      "Undamaged": 59.4
    }
  },
  "valid": {
    "images": 366,
    "boxes_per_class": {
      "Damage-detected": 138,
      "Undamaged": 272
    },
    "images_with_damage_box": 135,
    "images_undamaged_only": 230,
    "images_with_no_boxes": 1,
    "median_box_area_pct": {
      "Damage-detected": 64.0,
      "Undamaged": 60.0
    }
  },
  "test": {
    "images": 206,
    "boxes_per_class": {
      "Undamaged": 119,
      "Damage-detected": 92
    },
    "images_with_damage_box": 92,
    "images_undamaged_only": 113,
    "images_with_no_boxes": 1,
 

In [ ]:
import shutil
import os
from ultralytics import YOLO

MODEL = "yolo11s.pt"
EPOCHS = 30
IMGSZ = 640
BATCH = 16
QUICK_TEST = os.environ.get("QUICK_TEST", "0") == "1"   # real run by default

if QUICK_TEST:
    EPOCHS, FRACTION, IMGSZ, BATCH = 1, 0.05, 320, 4   # tiny run, only to test the code
else:
    FRACTION = 1.0

model = YOLO(MODEL)
model.train(
    data=str(root / "data.yaml"),
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, fraction=FRACTION,
    project=str(Path("runs").resolve()), name="damage", exist_ok=True,
)

Path("weights").mkdir(exist_ok=True)
shutil.copy("runs/damage/weights/best.pt", "weights/best.pt")

New https://pypi.org/project/ultralytics/8.4.189 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.173 🚀 Python-3.12.3 torch-2.11.0+cpu CPU (Intel Xeon Platinum 8370C CPU @ 2.80GHz)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/workspaces/daiist-assignment-2/data/damage-detect-v1/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=1, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=0.05, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=320, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mi

'weights/best.pt'

In [ ]:
import gc
import numpy as np
import pandas as pd

del model
gc.collect()

test_imgs = sorted(glob.glob(str(root / "test" / "images" / "*")))
if QUICK_TEST:
    test_imgs = test_imgs[:30]   # tiny slice, only to test the code


def has_damage(img_path):
    lab = root / "test" / "labels" / (Path(img_path).stem + ".txt")
    if not lab.exists():
        return 0
    return int(any(line.split()[0] == "0" for line in open(lab) if line.strip()))


y_true = np.array([has_damage(p) for p in test_imgs])


def damage_score(model, img_paths, class_ids):
    scores = []
    for r in model.predict(img_paths, conf=0.001, imgsz=IMGSZ, verbose=False, stream=True):
        s = [float(c) for c, k in zip(r.boxes.conf, r.boxes.cls) if int(k) in class_ids]
        scores.append(max(s) if s else 0.0)
    return np.array(scores)


def at_threshold(y, scores, t=0.5):
    pred = scores >= t
    tp = int((pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum())
    fn = int((~pred & (y == 1)).sum()); tn = int((~pred & (y == 0)).sum())
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "precision": round(tp / (tp + fp), 3) if tp + fp else 0.0,
            "recall": round(tp / (tp + fn), 3) if tp + fn else 0.0,
            "accuracy": round((tp + tn) / len(y), 3)}


# 1) Fine-tuned model
best = YOLO("weights/best.pt")
scores_ft = damage_score(best, test_imgs, {0})
pd.DataFrame({"image": [Path(p).name for p in test_imgs],
              "has_damage": y_true, "score": scores_ft}).to_csv("results/test_scores.csv", index=False)
if QUICK_TEST:
    map50 = map5095 = 0.0   # skipped in the quick test, computed in the real run
else:
    val = best.val(data=str(root / "data.yaml"), split="test", imgsz=IMGSZ,
                   batch=BATCH, verbose=False, plots=False)
    map50, map5095 = float(val.box.map50), float(val.box.map)
del best
gc.collect()

# 2) Zero-shot COCO model: no damage class exists, so it can never flag damage
coco = YOLO("yolo11s.pt")
vehicle_scores = damage_score(coco, test_imgs, {2, 5, 7})   # car, bus, truck
vehicle_share = float((vehicle_scores >= 0.25).mean())
nothing = np.zeros(len(y_true))

# 3) Trivial baseline: always say "not damaged"
results = {
    "finetuned": {**at_threshold(y_true, scores_ft),
                  "mAP50": round(map50, 3), "mAP50-95": round(map5095, 3)},
    "zero_shot_coco": {**at_threshold(y_true, nothing), "mAP50": 0.0, "mAP50-95": 0.0},
    "predict_nothing": {**at_threshold(y_true, nothing), "mAP50": 0.0, "mAP50-95": 0.0},
}
results["zero_shot_coco"]["share_of_test_images_where_it_sees_a_vehicle"] = round(vehicle_share, 3)
json.dump(results, open("results/baselines.json", "w"), indent=2)
print(pd.DataFrame(results).T)